In [ ]:
%pip install -q numpy matplotlib

## Scenario

## STAGE 1 -- CNNs for Image Data (55 min)

## STEP 1.1 -- Why CNNs? From MLPs to Convolutions (8 min)

## STEP 1.2 -- Exploring Image Data (8 min)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

# In practice, load a dataset like CIFAR-10 from your framework of choice.
# Images are 32x32 pixels with 3 color channels (RGB).
# Standard normalization uses per-channel means and standard deviations:
#   means = (0.4914, 0.4822, 0.4465)
#   stds  = (0.2470, 0.2435, 0.2616)
#
# For this conceptual walkthrough, we simulate an image batch:
SUBSET_SIZE = 5000
TEST_SUBSET = 1000

# Simulated image batch for shape demonstration
sample_images = np.random.rand(10, 32, 32, 3)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_images[i])
    ax.set_title(CLASSES[i % len(CLASSES)])
    ax.axis("off")
plt.suptitle("Sample Images (32x32x3)")
plt.tight_layout()
plt.show()

print(f"Image shape:        (3, 32, 32)  [channels, height, width]")
print(f"Training samples:   {SUBSET_SIZE}")
print(f"Test samples:       {TEST_SUBSET}")
print(f"Number of classes:  {len(CLASSES)}")

## STEP 1.3 -- Design a Basic CNN (15 min)

## STEP 1.4 -- CNN Training Concepts (10 min)

## STEP 1.5 -- Data Augmentation (8 min)

## STEP 1.6 -- Residual Connections (Stretch) (6 min)

## STAGE 2 -- RNNs for Sequence Data (55 min)

## STEP 2.1 -- From Images to Sequences (5 min)

## STEP 2.2 -- Generate Synthetic Sentiment Data (8 min)

In [ ]:
# STEP 2.2 -- Synthetic sentiment dataset
# Positive reviews: sequences with higher average token values
# Negative reviews: sequences with lower average token values

import numpy as np

VOCAB_SIZE = 50
SEQ_LEN = 20
NUM_TRAIN = 2000
NUM_TEST = 400

def generate_sentiment_data(n_samples, seq_len, vocab_size, seed=42):
    rng = np.random.RandomState(seed)
    sequences = []
    labels = []
    for _ in range(n_samples):
        label = rng.randint(0, 2)
        if label == 1:  # positive
            seq = rng.randint(vocab_size // 2, vocab_size, size=seq_len)
        else:           # negative
            seq = rng.randint(0, vocab_size // 2, size=seq_len)
        noise_idx = rng.choice(seq_len, size=seq_len // 4, replace=False)
        seq[noise_idx] = rng.randint(0, vocab_size, size=len(noise_idx))
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences), np.array(labels)

X_train_seq, y_train_seq = generate_sentiment_data(NUM_TRAIN, SEQ_LEN, VOCAB_SIZE, seed=42)
X_test_seq, y_test_seq = generate_sentiment_data(NUM_TEST, SEQ_LEN, VOCAB_SIZE, seed=99)

print(f"Training sequences shape: {X_train_seq.shape}")   # (2000, 20)
print(f"Test sequences shape:     {X_test_seq.shape}")     # (400, 20)
print(f"Vocabulary size:          {VOCAB_SIZE}")
print(f"Sequence length:          {SEQ_LEN}")
print(f"Class balance:            {y_train_seq.mean():.2f} positive")
print(f"\nSample sequence:          {X_train_seq[0]}")
print(f"Sample label:             {y_train_seq[0]}")

## STEP 2.3 -- Vanilla RNN for Sentiment (12 min)

## STEP 2.4 -- The Vanishing Gradient Problem in RNNs (8 min)

In [ ]:
# Demonstrate how gradient magnitudes decay across time steps
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
seq_length = 50
W_singular_value = 0.9

gradient_norms = []
current_magnitude = 1.0
for t in range(seq_length):
    gradient_norms.append(current_magnitude)
    current_magnitude *= W_singular_value

gradient_norms = gradient_norms[::-1]

plt.figure(figsize=(10, 4))
plt.bar(range(seq_length), gradient_norms)
plt.xlabel("Time Step (earlier → later)")
plt.ylabel("Relative Gradient Magnitude")
plt.title("Gradient Flow in Vanilla RNN (50 steps, σ_max = 0.9)")
plt.tight_layout()
plt.show()

print(f"Gradient at step 0 (earliest):  {gradient_norms[0]:.6f}")
print(f"Gradient at step 49 (latest):   {gradient_norms[49]:.6f}")
print(f"Ratio (earliest/latest):        {gradient_norms[0] / gradient_norms[49]:.6f}")

## STEP 2.5 -- LSTM: Gated Memory (12 min)

## STEP 2.6 -- GRU: Simplified Gating (5 min)

## STEP 2.7 -- Compare All Three Architectures (5 min)

In [ ]:
# Visualization of expected comparison (conceptual)
import matplotlib.pyplot as plt

epochs = list(range(1, 21))
# Simulated representative curves
losses_rnn  = [0.7 - 0.015*e + 0.002*e**0.5 for e in epochs]
losses_lstm = [0.7 - 0.025*e + 0.003*e**0.5 for e in epochs]
losses_gru  = [0.7 - 0.027*e + 0.003*e**0.5 for e in epochs]

accs_rnn  = [0.5 + 0.02*e - 0.0005*e**2 for e in epochs]
accs_lstm = [0.5 + 0.025*e - 0.0005*e**2 for e in epochs]
accs_gru  = [0.5 + 0.024*e - 0.0005*e**2 for e in epochs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, losses_rnn, label="Vanilla RNN")
ax1.plot(epochs, losses_lstm, label="LSTM")
ax1.plot(epochs, losses_gru, label="GRU")
ax1.set_title("Training Loss (Representative)"); ax1.set_xlabel("Epoch"); ax1.legend()

ax2.plot(epochs, accs_rnn, label="Vanilla RNN")
ax2.plot(epochs, accs_lstm, label="LSTM")
ax2.plot(epochs, accs_gru, label="GRU")
ax2.set_title("Test Accuracy (Representative)"); ax2.set_xlabel("Epoch"); ax2.legend()

plt.tight_layout(); plt.show()

## STAGE 3 -- Encoder-Decoder Architectures (45 min)

## STEP 3.1 -- The Encoder-Decoder Idea (8 min)

## STEP 3.2 -- Generate Sequence Reversal Data (5 min)

In [ ]:
# STEP 3.2 -- Generate digit-string reversal dataset
import numpy as np

PAD_TOKEN = 0
SOS_TOKEN = 11
EOS_TOKEN = 12
NUM_TOKENS = 13  # 0=PAD, 1-10=digits, 11=SOS, 12=EOS
MAX_LEN = 6

def generate_reversal_data(n_samples, max_len=MAX_LEN, seed=42):
    rng = np.random.RandomState(seed)
    encoder_inputs = []
    decoder_inputs = []
    decoder_targets = []

    for _ in range(n_samples):
        length = rng.randint(3, max_len + 1)
        seq = rng.randint(1, 11, size=length).tolist()
        reversed_seq = seq[::-1]

        enc_in = seq + [PAD_TOKEN] * (max_len - length)
        dec_in = [SOS_TOKEN] + reversed_seq + [PAD_TOKEN] * (max_len - length)
        dec_tgt = reversed_seq + [EOS_TOKEN] + [PAD_TOKEN] * (max_len - length)

        encoder_inputs.append(enc_in)
        decoder_inputs.append(dec_in)
        decoder_targets.append(dec_tgt)

    return (np.array(encoder_inputs),
            np.array(decoder_inputs),
            np.array(decoder_targets))

enc_train, dec_in_train, dec_tgt_train = generate_reversal_data(3000, seed=42)
enc_test, dec_in_test, dec_tgt_test = generate_reversal_data(500, seed=99)

print(f"Encoder input shape:  {enc_train.shape}")
print(f"Decoder input shape:  {dec_in_train.shape}")
print(f"Decoder target shape: {dec_tgt_train.shape}")
print(f"\nExample:")
print(f"  Encoder input:  {enc_train[0].tolist()}")
print(f"  Decoder input:  {dec_in_train[0].tolist()}")
print(f"  Decoder target: {dec_tgt_train[0].tolist()}")

## STEP 3.3 -- Design the Encoder (7 min)

## STEP 3.4 -- Design the Decoder (7 min)

## STEP 3.5 -- Assemble the Seq2Seq Model (5 min)

## STEP 3.6 -- Seq2Seq Training Concepts (8 min)

## STEP 3.7 -- Qualitative Evaluation (5 min)